# Deductible Comparison — deriving `$0` and `$250` from the canonical `$100`

Only the `$100` deductible is stored in the dataset. `$0` and `$250` are derived at runtime using methodology constants published in `pricing_config.deductible_handling`.

This notebook shows how to apply those rules yourself, including the Class D cap.

**License:** CC BY 4.0.

In [ ]:
import json
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/Optimal-Cover/vsc-pricing-data/main/data/v3.6.7"

# The JSON file keeps config_value as a native object (the CSV/Parquet stringify it).
config_raw = pd.read_json(f"{BASE_URL}/pricing_config.json")
config = {row['config_key']: row['config_value'] for _, row in config_raw.iterrows()}

deductible_rules = config['deductible_handling']['rules']
print(json.dumps(deductible_rules, indent=2))

## 1. Implement the derivation

Rules from the methodology:
- `$0`  → multiply $100 price by `1.35`, round to nearest `$50`, cap at `+$500` for Class D
- `$250` → multiply $100 price by `0.85`, round to nearest `$50`

In [ ]:
def round_to(value, increment):
    return round(value / increment) * increment

def derive_price(canonical_100, target_deductible, vehicle_class):
    rule = deductible_rules[str(target_deductible)]
    derived = canonical_100 * rule['multiplier']
    derived = round_to(derived, rule['rounding']['increment'])

    # Apply any caps
    for cap in rule.get('caps', []):
        if vehicle_class in cap['scope']['vehicle_class']:
            if cap['cap_type'] == 'ABSOLUTE_ADDITIVE':
                max_allowed = canonical_100 + cap['cap_amount']
                derived = min(derived, max_allowed)

    return int(derived)

## 2. Apply to the full rates table

In [ ]:
rates = pd.read_csv(f"{BASE_URL}/vsc_rates.csv")

rates['price_0']   = rates.apply(lambda r: derive_price(r['price'], 0,   r['vehicle_class']), axis=1)
rates['price_100'] = rates['price']
rates['price_250'] = rates.apply(lambda r: derive_price(r['price'], 250, r['vehicle_class']), axis=1)

rates[['vehicle_class', 'mileage_band', 'term_months', 'band',
       'price_0', 'price_100', 'price_250']].head(12)

## 3. Verify the Class D cap

For Class D vehicles the `$0` surcharge over the `$100` price is capped at `+$500`. Anywhere the uncapped `1.35×` would exceed that ceiling, the cap kicks in.

In [ ]:
d = rates[rates['vehicle_class'] == 'D'].copy()
d['uncapped_0'] = (d['price'] * 1.35).round(-1).astype(int)
d['implied_cap_hit'] = d['uncapped_0'] > (d['price'] + 500)

d[d['implied_cap_hit']][['mileage_band', 'term_months', 'band',
                          'price', 'uncapped_0', 'price_0']].head(10)

## 4. Worked example

A Class C vehicle, 60,001–72,000 mi band, 48-month term, `REFERENCE_HIGH`:

In [ ]:
mask = (
    (rates['vehicle_class'] == 'C') &
    (rates['mileage_band'] == '60,001 - 72,000') &
    (rates['term_months'] == 48) &
    (rates['band'] == 'REFERENCE_HIGH')
)
rates.loc[mask, ['price_0', 'price_100', 'price_250']]

---

Only the `$100` column is the canonical authority. The other two are deterministic derivations — anyone applying the same `pricing_config` rules should get exactly the same numbers.